# Datos de clorofila A de PACE sobre una región del océano Atlántico Norte

Este notebook demuestra el acceso y el subconjunto de datos PACE Ocean Color. Se puede encontrar información general sobre el dataset en el sitio web de PACE (ver [aquí](https://oceandata.sci.gsfc.nasa.gov)).

**Requisitos para ejecutar este notebook**
1. Tener una cuenta Earth Data Login
2. Tener un Bearer Token.
3. Conocer un collection concept ID (o DOI) de un dataset.
4. Conexión a internet.


**Objetivos**

Usar [pydap](https://pydap.github.io/pydap/) para demostrar:

- Descubrimiento de gránulos OPeNDAP desde una colección de interés, con filtrado adicional por rango temporal.
- Uso de `tokens` para autenticación (desde earthaccess).
- Acceso a `PACE` Nivel 3 e identificación de un área de interés.
- Subconjunto y habilitación de la `Constraint Expression` para que llegue al servidor OPeNDAP.
- Almacenar el subconjunto localmente.

`Autor`: Miguel Jimenez-Urias, '25


In [ ]:
from pydap.client import get_cmr_urls, consolidate_metadata, open_url
from pydap.net import create_session
import xarray as xr
import earthaccess
import datetime as dt
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors

### Usar una sesión como base de datos de metadatos y autenticación

Pydap puede usar un objeto `requests_cache.CachedSession`, que puede servir como:

- Autenticación (mediante netrc o token).
- Gestión de base de datos (sqlite3 como backend). Solo cachea los datos necesarios.
- Transmisión de datos.

Abajo inicializamos la sesión, heredando credenciales de autenticación desde earthaccess, y apuntando a la base de datos local `PACE.sqlite` almacenada en la carpeta `/data`.


In [ ]:
auth = earthaccess.login(strategy="netrc", persist=True)
session = create_session(use_cache=True, session=earthaccess.get_requests_https_session(), cache_kwargs={'cache_name': 'data/PACE'})
session.cache.clear()

### Consultar CMR para URLs OPeNDAP

Consultamos el CMR de NASA para todas las URLs opendap asociadas con Chlorophyll A mallado, diario, Nivel 3, a 4 km, versión 3.1. Para esto, puedes usar Earthdata Search para identificar el Concept Collection ID. Nos enfocamos en 2 meses de datos de este año.


In [ ]:
PACE_ccid = "C3620140255-OB_CLOUD" # version 3.1
time_range=[dt.datetime(2025, 6, 1), dt.datetime(2025, 8, 1)]
urls = get_cmr_urls(ccid=PACE_ccid, limit=1000, time_range=time_range) # limit by default = 50


```{note}
Esta colección tenía dos versiones de los mismos datos. Nos interesa la resolución `4km`. Una forma de asegurarse de que cualquier número de URLs opendap pertenecen a la misma colección es inspeccionando las cadenas de URL (innecesario en la mayoría de los casos).
```


Además, subdividimos por nombre de variable, reteniendo solo las variables `lat`, `lon` y `chlor_a`. Estas son las variables de interés, y descartar variables reduce el tiempo que tarda `Xarray` en procesar los metadatos.


```{note}
Xarray tiene un método .drop_vars, pero estas variables se descartan DESPUÉS de crear el dataset. Si un dataset tiene O(1000) variables, `Xarray` procesaría primero las variables y luego las descartaría. Con OPeNDAP, puedes indicar al servidor de antemano qué variables necesita procesar Xarray.

```


In [ ]:
CEs = "?dap4.ce=/lat;/lon;/chlor_a"

In [ ]:
_urls = [url for url in urls if '4km' in url and "DAY" in url]
pace_urls = [url.replace("https", "dap4") + CEs for url in _urls] # a dap4 schema implies a DAP4 protocol in pydap.
len(pace_urls)

In [ ]:
pace_urls[:4]

In [ ]:
session=create_session()

### Crear un único objeto Dataset

Usamos `Xarray` con `pydap` como backend. `Pydap` es un backend que siempre se instala al instalar `Xarray`, pero aún no es el predeterminado al trabajar con URLs opendap. Por eso debemos definirlo.


In [ ]:
%%time
ds = xr.open_mfdataset(
    pace_urls, 
    engine='pydap', 
    session=session, 
    parallel=True, 
    concat_dim='time',  # <------ a time dimension will be created 
    combine='nested',
)

In [ ]:
ds

El dataset anterior representa una representación de metadatos de todos los datos de interés. Está fragmentado, y solo se han descargado las dimensiones lat y lon.

```{note}
Se creó una dimensión con nombre `time`, junto con todos los gránulos remotos concatenados.
```


In [ ]:
print('Uncompressed dataset size [GBs]: ', ds.nbytes / 1e9)

### Subconjunto próximo a los datos: ¿cómo hacerlo con OPeNDAP, Pydap y Xarray?


Aunque es posible descargar todo el dataset, es mejor práctica subdividirlo primero. Hay muchas herramientas para subdividir datos de forma próxima a los datos, y OPeNDAP es una de ellas. Necesitamos identificar primero el subconjunto y pasarlo al servidor.

Identificar el subconjunto permite construir potencialmente Constraint Expressions, que se usan para indicar al servidor OPeNDAP el subconjunto de interés. Pero con Pydap y Xarray, estas herramientas construyen rebanados de forma interactiva y pueden solicitar subconjuntos, ocultando la abstracción. Abajo demostramos cómo pasar el subconjunto al servidor OPeNDAP para que Xarray haga menos trabajo.

### Identificar subconjunto espacial

En este caso nos interesa un subconjunto espacial. Los datos son Nivel 3 (mallados), por lo que latitud y longitud son uniformes. Además, son 1D y ya se descargaron en memoria.


In [ ]:
lat, lon = ds['lat'].values, ds['lon'].values

### Supongamos que definimos el área de interés


In [ ]:
# Min/max of lon values
minLon, maxLon = -96, 10

# Min/Max of lat values
minLat, maxLat = 6, 70

In [ ]:
iLon = np.where((lon>minLon)&(lon < maxLon))[0]
iLat= np.where((lat>minLat)&(lat < maxLat))[0]

Así que los rebanados son simplemente el primer y último elemento de iLon e iLat:

```python
lon_slice = slice(iLon[0], iLon[-1])
lat_slice = slice(iLat[0], iLat[-1])
```


```{warning}
No todos los valores de arreglos de coordenadas `lat/lon` son monotónicos. Asegúrate siempre de que lo sean, incluso cuando los datos sean Nivel 3 o Nivel 4.

```


### Descargar solo el subconjunto


Con `Xarray`, la sintaxis de subconjunto es similar a pandas. Por ejemplo:
```
ds.isel(lon=lon_slice, lat_slice)
```

Produce un subconjunto del lado del cliente. Sin embargo:

```{warning}
Cuando se usa opendap, rebanar como arriba y luego descargar datos solo subdividirá por variable, descargando en la mayoría de los casos todos los datos, para después volver a subdividirlos con xarray.
```


Para asegurar que los rebanados se envíen al servidor remoto, debemos aplicar `chunk the dataset` al crearlo. Esto garantizará que el subconjunto se haga principalmente del lado del servidor. `DEBES` elegir el tamaño de chunk para que coincida con el tamaño del rebanado, como se muestra abajo.


In [ ]:
%%time
ds = xr.open_mfdataset(
    pace_urls, 
    engine='pydap', 
    session=session, 
    parallel=True, 
    concat_dim='time',  # <------ a time dimension will be created 
    combine='nested',
    chunks = {'lon': len(iLon), 'lat':len(iLat)} #  <----------- This instructs the OPeNDAP server to subset in space
)
ds

In [ ]:
ds["chlor_a"] ## inspect the chunk of the data

Ahora nos aseguramos de que `Xarray` trate este subconjunto como un chunk individual (por variable).


In [ ]:
ds = ds.isel(lon=slice(iLon[0], iLon[-1]+1), lat=slice(iLat[0], iLat[-1]+1))

### Finalmente

Almacenar datos localmente como `NetCDF4`. En esta etapa:

- Los datos se descargan (subdivididos espacialmente mediante `OPeNDAP`)
- Los datos se remuestrean (mediante `Xarray`)


In [ ]:
%%time
ds.to_netcdf("data/pace_subset.nc4", mode='w')

### Inspeccionamos nuestros datos


In [ ]:
mds = xr.open_dataset("data/pace_subset.nc4")
mds

In [ ]:
print("Size of downloaded subset: ", mds.nbytes/1e9)